In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import os

# --- CONFIGURATION ---
class Config:
    # INPUT: Your raw sequence file
    data_path = 'reduced_256_embedd.pkl' 
    
    # OUTPUT: The new file to save
    output_path = 'finalized_pooled_embeddings.pkl'

    # Training params for the Attention mechanism
    epochs = 500           # Doesn't need 1000, just enough to learn attention
    batch_size = 256
    lr = 5e-4
    
    # Encoder Params (Must match your desired architecture)
    input_dim = 384      # CHANGE THIS to your raw embedding size (e.g. 1280 for ESM-2, 1024 for ProtT5)
    seq_hidden = 256      # The size of the output vector
    seq_layers = 1
    seq_dropout = 0.1
    n_head = 16
    
    # Data params
    max_seq_len = 256
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 1. DATASET & UTILS (From your script)
# ==========================================

class MutationDataset(Dataset):
    def __init__(self, pkl_path, max_seq_len=1024):
        print(f"Loading {pkl_path}...")
        
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)

        if hasattr(data, 'loc'):
            self.fitness = data['fitness'].values
            embeddings_raw = data['embedding'].values
            self.file_names = data['file_name'].values
        else:
            self.fitness = np.array(data['fitness'])
            embeddings_raw = np.array(data['embedding'])
            self.file_names = np.array(data['file_name'])

        # Auto-detect input dimension from first sample
        self.raw_dim = embeddings_raw[0].shape[-1]
        Config.input_dim = self.raw_dim
        print(f"Detected Input Feature Dimension: {self.raw_dim}")

        self.max_h = max_seq_len
        
        # Determine max width (feature dim)
        if len(embeddings_raw[0].shape) > 1:
            self.max_w = embeddings_raw[0].shape[1]
        else:
            self.max_w = 1

        self.embeddings = np.stack(embeddings_raw).astype(np.float32)
        self.lengths = np.array([e.shape[0] for e in self.embeddings], dtype=np.int32)
        # Normalize
        mean = self.embeddings.mean(axis=(0, 1))
        std = self.embeddings.std(axis=(0, 1)) + 1e-6
        self.embeddings = (self.embeddings - mean[None, None, :]) / std[None, None, :]

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return {
            'embedding': torch.tensor(self.embeddings[idx], dtype=torch.float32),
            'length': int(self.lengths[idx]),
            'fitness': torch.tensor(self.fitness[idx], dtype=torch.float32),
            'file_name': str(self.file_names[idx])
        }

def collate_fn(batch):
    embs = torch.stack([b['embedding'] for b in batch])
    lengths = torch.tensor([b['length'] for b in batch], dtype=torch.long)
    fitness = torch.stack([b['fitness'] for b in batch])
    names = [b['file_name'] for b in batch]
    return embs, lengths, fitness, names

# ==========================================
# 2. SEQUENCE ENCODER (Exactly as requested)
# ==========================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) 
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class SequenceEncoder(nn.Module):
    def __init__(self, input_dim, hidden_size=256, num_layers=2, dropout=0.1, nhead=8):
        super().__init__()
        assert hidden_size % nhead == 0, "hidden_size must be divisible by nhead"

        self.hidden_size = hidden_size

        self.input_proj = nn.Linear(input_dim, hidden_size)
        self.pos_encoder = PositionalEncoding(hidden_size, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=nhead,
            dim_feedforward=hidden_size * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Attention Pooling Head
        self.attn_proj = nn.Linear(hidden_size, hidden_size // 2)
        self.attn_vec = nn.Linear(hidden_size // 2, 1)

    def forward(self, x, lengths):
        device = x.device
        batch_size, seq_len, _ = x.shape
        
        # Masking
        padding_mask = torch.arange(seq_len, device=device).unsqueeze(0) >= lengths.unsqueeze(1)

        x = self.input_proj(x)
        x = self.pos_encoder(x)
        out = self.transformer(x, src_key_padding_mask=padding_mask)

        # Attention Pooling
        proj = torch.tanh(self.attn_proj(out))
        scores = self.attn_vec(proj).squeeze(-1)
        scores = scores.masked_fill(padding_mask, float('-inf'))
        attn = torch.softmax(scores, dim=-1).unsqueeze(-1)
        
        pooled = torch.sum(out * attn, dim=1)
        return pooled

# ==========================================
# 3. TRAINING WRAPPER
# ==========================================

class PoolingTrainer(nn.Module):
    """
    Wraps the SequenceEncoder with a temporary head to allow gradient flow.
    We train this to minimize fitness prediction error, which teaches
    the attention mechanism WHICH amino acids matter.
    """
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        # Simple regression head
        self.head = nn.Linear(encoder.hidden_size, 1)

    def forward(self, x, lengths):
        pooled = self.encoder(x, lengths)
        return self.head(pooled).squeeze()

# ==========================================
# 4. MAIN SCRIPT
# ==========================================

def main():
    print(f"--- PRE-TRAINING ATTENTION POOLING ---")
    
    # 1. Load Data
    dataset = MutationDataset(Config.data_path, max_seq_len=Config.max_seq_len)
    loader = DataLoader(dataset, batch_size=Config.batch_size, shuffle=True, collate_fn=collate_fn)
    
    # 2. Initialize Model
    seq_encoder = SequenceEncoder(
        input_dim=Config.input_dim,
        hidden_size=Config.seq_hidden,
        num_layers=Config.seq_layers,
        dropout=Config.seq_dropout,
        nhead=Config.n_head
    ).to(Config.device)

    model = PoolingTrainer(seq_encoder).to(Config.device)
    
    # 3. Train the Attention Mechanism
    # We use HuberLoss (Regression) because it provides a granular gradient 
    # to learn the attention weights better than classification labels.
    optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr)
    criterion = nn.HuberLoss()

    print(f"\nTraining Pooling Mechanism for {Config.epochs} epochs...")
    model.train()
    
    best_loss = 100
    break_cicle = 0
    for epoch in range(Config.epochs):
        total_loss = 0
        for embs, lengths, fitness, _ in loader:
            embs, lengths, fitness = embs.to(Config.device), lengths.to(Config.device), fitness.to(Config.device)
            
            optimizer.zero_grad()
            preds = model(embs, lengths)
            loss = criterion(preds, fitness)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        epoch_loss = (total_loss/len(loader))

        if ((epoch+1) % 10) == 0 or epoch == 0:
            print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

        if best_loss > epoch_loss : 
            best_loss = epoch_loss
            break_cicle = 0
        else:
            break_cicle += 1
        
        if break_cicle == 10 : break

        

    # ==========================================
    # 5. EXTRACT AND SAVE
    # ==========================================
    print("\n--- SAVING POOLED EMBEDDINGS ---")
    
    # Create a loader that doesn't shuffle, to preserve order if needed (though we save file_name)
    save_loader = DataLoader(dataset, batch_size=Config.batch_size, shuffle=False, collate_fn=collate_fn)
    
    seq_encoder.eval()
    
    final_embeddings = []
    final_fitness = []
    final_names = []

    with torch.no_grad():
        for embs, lengths, fitness, names in tqdm(save_loader, desc="Extracting"):
            embs = embs.to(Config.device)
            lengths = lengths.to(Config.device)
            
            # Extract just the pooled vector (ignore the temporary regression head)
            pooled = seq_encoder(embs, lengths)
            
            final_embeddings.append(pooled.cpu().numpy())
            final_fitness.append(fitness.numpy())
            final_names.extend(names)

    # Concatenate
    all_embs = np.concatenate(final_embeddings, axis=0)
    all_fits = np.concatenate(final_fitness, axis=0)
    
    # Save Dictionary
    save_data = {
        'embedding': all_embs, # Shape: (N, 256)
        'fitness': all_fits,   # Shape: (N,)
        'file_name': final_names
    }
    
    print(f"Saving to {Config.output_path}...")
    with open(Config.output_path, 'wb') as f:
        pickle.dump(save_data, f)
        
    print(f"Done! New data shape: {all_embs.shape}")
    print("You can now run your classification script on 'finalized_pooled_embeddings.pkl'")

if __name__ == "__main__":
    main()